# 04  -  Reconcile predicted vs. actual cost

This notebook joins the latest `run_log` entries to [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html) and shows predicted vs. actual cost side-by-side for both AI Function paths.

**This notebook is idempotent and safe to re-run.** It makes no `ai_extract` or `ai_query` calls  -  only reads existing tables. Cost per refresh: essentially \$0. Schedule it hourly to track the variance as billing data populates.

**What this notebook does:**
1. Reads the latest `run_log` entry for each path (`ai_extract`, `ai_query`).
2. **Freshness gate:** counts billing rows in the run window. If zero, prints `"NO BILLING DATA YET"` and exits cleanly.
3. Joins to `system.billing.usage` filtered by run window + `usage_metadata.endpoint_name` (`databricks-ai-extract` for `ai_extract`, the configured `model_name` for `ai_query`). This is the only reliable attribution because `billing_origin_product = 'AI_FUNCTIONS'` is **not always populated** in the 2026 schema  -  both paths frequently bill under `MODEL_SERVING / PREMIUM_SERVERLESS_REAL_TIME_INFERENCE` differentiated only by endpoint.
4. Joins to `system.billing.list_prices` so usage is converted to USD using the price in effect when the call ran.
5. Outputs three displays:
   - **Per-bucket cost breakdown** (where the money landed).
   - **Per-path estimate vs. actual** (the headline).
   - **Per-row cost** (extrapolation for the customer's full workload).

**Public references:**
- [System tables: billing](https://docs.databricks.com/en/admin/system-tables/billing.html)
- [Cost queries for AI Functions on Azure](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-functions#view-costs-for-ai-function-workloads)
- [Budget alerts (recommended next step)](https://docs.databricks.com/en/admin/account-settings/budgets.html)

**Note on billing lag:** [`system.billing.usage` typically lags 1-4 hours](https://docs.databricks.com/en/admin/system-tables/billing.html#:~:text=lag) behind real usage  -  this is documented behavior. Re-run this notebook hourly until the actual cost row appears.


## Read job parameters


In [ ]:
import datetime as dt
from pyspark.sql import functions as F

dbutils.widgets.text("catalog",      "main")
dbutils.widgets.text("schema",       "ai_functions_cost_demo")
dbutils.widgets.text("team_tag",     "ai-cost-demo")
dbutils.widgets.text("owner_email",  "you@example.com")
dbutils.widgets.text("model_name",   "databricks-meta-llama-3-3-70b-instruct")

CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
TEAM_TAG    = dbutils.widgets.get("team_tag")
OWNER_EMAIL = dbutils.widgets.get("owner_email")
MODEL_NAME  = dbutils.widgets.get("model_name")  # endpoint name for ai_query (full FM endpoint id)

# Endpoints we filter on in system.billing.usage. ai_extract always lives on the managed
# `databricks-ai-extract` endpoint; ai_query lives on whichever FM endpoint was called.
EXTRACT_ENDPOINT = "databricks-ai-extract"
QUERY_ENDPOINT   = MODEL_NAME

RUN_LOG_TABLE = f"{CATALOG}.{SCHEMA}.run_log"
print(f"Reading run log: {RUN_LOG_TABLE}")
print(f"Owner filter:    {OWNER_EMAIL}")
print(f"Endpoints:       ai_extract -> {EXTRACT_ENDPOINT}")
print(f"                 ai_query   -> {QUERY_ENDPOINT}")


## Load latest run for each path

The next cell pulls the most recent `run_log` row per path (`ai_extract`, `ai_query`). The reconciliation window is taken from the earliest `run_start` and the latest `run_end` across both paths, widened by ±1 hour to catch billing rows that straddle the boundary.


In [ ]:
# Latest run per path
latest_runs = spark.sql(f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY path ORDER BY logged_ts DESC) AS rn
        FROM {RUN_LOG_TABLE}
        WHERE owner = '{OWNER_EMAIL}'
          AND team  = '{TEAM_TAG}'
    )
    WHERE rn = 1
""").collect()

if not latest_runs:
    raise Exception(f"No runs found in {RUN_LOG_TABLE} for owner={OWNER_EMAIL} team={TEAM_TAG}. Run the full pipeline first.")

print(f"Found {len(latest_runs)} latest runs:")
for r in latest_runs:
    print(f"  {r['path']:>12}  run_id={r['run_id']}  rows={r['n_rows']:>5}  start={r['run_start']}  end={r['run_end']}")

# Reconciliation window: earliest start to latest end, widened by ±1h
window_start = min(r["run_start"] for r in latest_runs)
window_end   = max(r["run_end"]   for r in latest_runs)
print(f"\nReconciliation window: {window_start} -> {window_end}")


## Freshness gate

The next cell checks whether `system.billing.usage` has any rows in the run window. If zero, prints a clear status and stops the notebook gracefully  -  that's the expected behavior for the first 1-4 hours after a run.

**If you see `NO BILLING DATA YET`:** wait an hour and re-run. The hourly-scheduled job (`ai-functions-cost-reconcile-hourly`) does this automatically.


In [ ]:
freshness = spark.sql(f"""
    SELECT COUNT(*) AS billing_rows
    FROM system.billing.usage AS u
    WHERE u.usage_start_time < TIMESTAMP'{window_end}'   + INTERVAL 1 HOUR
      AND u.usage_end_time   > TIMESTAMP'{window_start}' - INTERVAL 1 HOUR
      AND u.billing_origin_product = 'MODEL_SERVING'
      AND u.usage_metadata.endpoint_name IN ('{EXTRACT_ENDPOINT}', '{QUERY_ENDPOINT}')
""").collect()[0]
billing_rows = freshness["billing_rows"]

now_utc = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
hours_since_end = (now_utc - window_end.replace(tzinfo=None)).total_seconds() / 3600

print(f"Hours since latest run end: {hours_since_end:.2f}")
print(f"Billing rows in window:     {billing_rows}")
print()

billing_ready = billing_rows > 0
if not billing_ready:
    print("STATUS: NO BILLING DATA YET")
    print("  system.billing.usage typically lags 1-4 hours behind real usage.")
    print("  The hourly-scheduled job (ai-functions-cost-reconcile-hourly) will re-run this notebook on the hour.")
    print("  This run is exiting cleanly with no further work.")
else:
    print(f"STATUS: BILLING DATA AVAILABLE ({billing_rows} rows). Reconciling...")


## Per-endpoint cost breakdown

If billing is available, the next cell shows the actual cost split by `usage_metadata.endpoint_name`. In the 2026 schema both function families typically land under `MODEL_SERVING / PREMIUM_SERVERLESS_REAL_TIME_INFERENCE`, so the **endpoint is the attribution key**, not `billing_origin_product`:
- `databricks-ai-extract`  -  the managed endpoint backing every `ai_extract` call
- `databricks-meta-llama-3-3-70b-instruct` (or whatever `model_name` is set to)  -  the FM endpoint backing `ai_query`

Re-running on a window with no billing rows prints an empty table  -  that's expected before lag clears.


In [ ]:
if billing_ready:
    breakdown = spark.sql(f"""
        SELECT
            CASE
                WHEN u.usage_metadata.endpoint_name = '{EXTRACT_ENDPOINT}' THEN 'A. ai_extract endpoint (databricks-ai-extract)'
                WHEN u.usage_metadata.endpoint_name = '{QUERY_ENDPOINT}'   THEN CONCAT('B. ai_query endpoint (', u.usage_metadata.endpoint_name, ')')
                ELSE CONCAT('Other: ', u.usage_metadata.endpoint_name)
            END AS bucket,
            u.usage_metadata.endpoint_name AS endpoint,
            u.sku_name,
            ROUND(SUM(u.usage_quantity), 4) AS dbus,
            ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0.07)), 4) AS cost_usd,
            COUNT(*) AS billing_rows
        FROM system.billing.usage u
        LEFT JOIN system.billing.list_prices lp
          ON u.sku_name = lp.sku_name
          AND u.cloud   = lp.cloud
          AND lp.currency_code = 'USD'
          AND u.usage_start_time >= lp.price_start_time
          AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
        WHERE u.usage_start_time < TIMESTAMP'{window_end}'   + INTERVAL 1 HOUR
          AND u.usage_end_time   > TIMESTAMP'{window_start}' - INTERVAL 1 HOUR
          AND u.billing_origin_product = 'MODEL_SERVING'
          AND u.usage_metadata.endpoint_name IN ('{EXTRACT_ENDPOINT}', '{QUERY_ENDPOINT}')
        GROUP BY 1, 2, 3
        ORDER BY cost_usd DESC
    """)
    print("Actual billing breakdown:")
    display(breakdown)


## Per-path estimate vs. actual  -  the headline table

The next cell builds one row per path comparing predicted vs. actual:

| path | rows | predicted_low_list | predicted_medium_list | predicted_token_reference | actual_usd | variance_pct | variance_explanation |
|---|---|---|---|---|---|---|---|

Variance bands and explanations:
- `actual` is null → "No billing data for this path yet (1-4h lag)  -  re-run later"
- `actual` within 20% of predicted → "Estimate close (within 20%)"
- `actual` is between 1.2× and 5× predicted → "Significant gap  -  check DBU rates / complexity bracket"
- `actual` > 5× predicted → "Likely on a different pricing tier than estimated"

This is the row to put on a slide.


In [ ]:
if billing_ready:
    # Build a per-path actuals query, then union the predictions per path
    parts = []
    for r in latest_runs:
        path = r["path"]
        # Per-path filter on endpoint_name (the only reliable way to attribute billing
        # to a specific AI Function call  -  billing_origin_product alone is too broad and
        # AI_FUNCTIONS bucket is not always populated in 2026 schema).
        if path == "ai_extract":
            bucket_clause = (
                f"u.billing_origin_product = 'MODEL_SERVING' "
                f"AND u.usage_metadata.endpoint_name = '{EXTRACT_ENDPOINT}'"
            )
            pred_low      = r["predicted_low_list"]
            pred_med      = r["predicted_medium_list"]
            pred_tok      = None
        else:  # ai_query
            bucket_clause = (
                f"u.billing_origin_product = 'MODEL_SERVING' "
                f"AND u.usage_metadata.endpoint_name = '{QUERY_ENDPOINT}'"
            )
            pred_low      = None
            pred_med      = None
            pred_tok      = r["predicted_token_reference"]
        parts.append({
            "path": path, "n_rows": r["n_rows"],
            "run_start": r["run_start"], "run_end": r["run_end"],
            "predicted_low_list": pred_low,
            "predicted_medium_list": pred_med,
            "predicted_token_reference": pred_tok,
            "bucket_clause": bucket_clause,
        })

    sql_unions = []
    for p in parts:
        sql_unions.append(f"""
        SELECT
            '{p['path']}' AS path,
            {p['n_rows']} AS rows,
            CAST({p['predicted_low_list'] if p['predicted_low_list'] is not None else 'NULL'} AS DOUBLE)        AS predicted_low_list,
            CAST({p['predicted_medium_list'] if p['predicted_medium_list'] is not None else 'NULL'} AS DOUBLE)  AS predicted_medium_list,
            CAST({p['predicted_token_reference'] if p['predicted_token_reference'] is not None else 'NULL'} AS DOUBLE) AS predicted_token_reference,
            (SELECT ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0.07)), 4)
             FROM system.billing.usage u
             LEFT JOIN system.billing.list_prices lp
               ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud AND lp.currency_code = 'USD'
               AND u.usage_start_time >= lp.price_start_time
               AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
             WHERE u.usage_start_time < TIMESTAMP'{p['run_end']}'   + INTERVAL 1 HOUR
               AND u.usage_end_time   > TIMESTAMP'{p['run_start']}' - INTERVAL 1 HOUR
               AND u.custom_tags.Owner = '{OWNER_EMAIL}'
               AND {p['bucket_clause']}
            ) AS actual_usd
        """)

    headline_sql = " UNION ALL ".join(sql_unions)
    headline = spark.sql(f"""
        WITH base AS ({headline_sql})
        SELECT
            path, rows,
            ROUND(predicted_low_list, 4)        AS pred_low_list,
            ROUND(predicted_medium_list, 4)     AS pred_med_list,
            ROUND(predicted_token_reference, 4) AS pred_token_ref,
            actual_usd,
            CASE
                WHEN path = 'ai_extract' AND predicted_low_list IS NOT NULL AND actual_usd IS NOT NULL
                  THEN ROUND((actual_usd - predicted_low_list) / predicted_low_list * 100, 1)
                WHEN path = 'ai_query'   AND predicted_token_reference IS NOT NULL AND actual_usd IS NOT NULL
                  THEN ROUND((actual_usd - predicted_token_reference) / predicted_token_reference * 100, 1)
                ELSE NULL
            END AS variance_pct,
            CASE
                WHEN actual_usd IS NULL THEN 'No billing data yet for this endpoint (1-4h lag)  -  re-run later'
                WHEN path = 'ai_extract' AND predicted_low_list IS NOT NULL THEN
                    CASE
                        WHEN ABS(actual_usd - predicted_low_list) / predicted_low_list <= 0.2 THEN 'Estimate close (within 20%)'
                        WHEN actual_usd < predicted_low_list * 0.5 THEN 'Predicted too high  -  per-input formula assumes AI_FUNCTIONS DBU rate; actual endpoint runs cheaper'
                        WHEN actual_usd > predicted_low_list * 5 THEN 'Predicted too low  -  likely a higher complexity bracket than "low"'
                        ELSE 'Check complexity bracket / DBU rate'
                    END
                WHEN path = 'ai_query' AND predicted_token_reference IS NOT NULL THEN
                    CASE
                        WHEN ABS(actual_usd - predicted_token_reference) / predicted_token_reference <= 0.2 THEN 'Estimate close (within 20%)'
                        WHEN actual_usd > predicted_token_reference * 5 THEN 'Token estimate is a floor  -  MODEL_SERVING bills compute time, not tokens; expect undershoot at small N'
                        WHEN actual_usd < predicted_token_reference * 0.5 THEN 'Token estimate overshot  -  verify model and prompt sizes'
                        ELSE 'Check model rates / overhead'
                    END
                ELSE NULL
            END AS variance_explanation
        FROM base
        ORDER BY path
    """)
    print("Per-path estimate vs actual:")
    display(headline)


## Per-row cost  -  extrapolate to customer scale

The next cell divides the actual USD cost by the number of rows in each run, giving the per-row cost. The customer can multiply this by their actual daily/monthly row volume (e.g. 4,723 rows/day × 30 days = 141,690 rows/month) for a realistic forecast based on their *measured* cost on this exact workload.

This is the most actionable number for the meeting  -  much more honest than the pricing-page estimate because it reflects the customer's actual transcript complexity.


In [ ]:
if billing_ready:
    perrow = spark.sql(f"""
        WITH actuals AS (
            SELECT
                CASE
                    WHEN u.usage_metadata.endpoint_name = '{EXTRACT_ENDPOINT}' THEN 'ai_extract'
                    WHEN u.usage_metadata.endpoint_name = '{QUERY_ENDPOINT}'   THEN 'ai_query'
                END AS path,
                ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0.07)), 4) AS actual_usd
            FROM system.billing.usage u
            LEFT JOIN system.billing.list_prices lp
              ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud AND lp.currency_code = 'USD'
              AND u.usage_start_time >= lp.price_start_time
              AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
            WHERE u.usage_start_time < TIMESTAMP'{window_end}'   + INTERVAL 1 HOUR
              AND u.usage_end_time   > TIMESTAMP'{window_start}' - INTERVAL 1 HOUR
              AND u.billing_origin_product = 'MODEL_SERVING'
              AND u.usage_metadata.endpoint_name IN ('{EXTRACT_ENDPOINT}', '{QUERY_ENDPOINT}')
            GROUP BY 1
        ),
        runs AS (
            SELECT path, n_rows
            FROM (
                SELECT path, n_rows, ROW_NUMBER() OVER (PARTITION BY path ORDER BY logged_ts DESC) rn
                FROM {RUN_LOG_TABLE}
                WHERE owner = '{OWNER_EMAIL}' AND team = '{TEAM_TAG}'
            ) WHERE rn = 1
        )
        SELECT
            r.path, r.n_rows,
            a.actual_usd,
            ROUND(a.actual_usd / r.n_rows, 6) AS cost_per_row,
            ROUND(a.actual_usd / r.n_rows * 4723, 2)         AS extrapolated_4723_rows_day,
            ROUND(a.actual_usd / r.n_rows * 4723 * 30, 2)    AS extrapolated_4723_rows_month
        FROM runs r
        LEFT JOIN actuals a ON r.path = a.path
        ORDER BY r.path
    """)
    print("Per-row actual cost + extrapolation to 4,723 rows/day:")
    display(perrow)


## Done

If actuals were available, you have three displays above:
1. **Per-bucket breakdown**  -  proves the two pricing models live in different `billing_origin_product` values.
2. **Per-path estimate vs actual**  -  the headline variance row for the meeting slide.
3. **Per-row + extrapolation to 4,723 rows/day**  -  the customer's realistic monthly forecast.

If actuals were not yet available (`STATUS: NO BILLING DATA YET`), the hourly-scheduled job will re-run this notebook automatically until billing populates.

**Recommended next steps for the customer:**
1. Tag the production fraud-detection job with `custom_tags.team = 'fraud-detection'` and `cost_center = '<id>'` so spend rolls up by team in `system.billing.usage`.
2. Set a [Budget Alert](https://docs.databricks.com/en/admin/account-settings/budgets.html) at ~1.5x the per-day forecast above.
3. Pick the cheaper path that still meets quality. If both `ai_extract` and `ai_query` produce equivalent fraud-signal extraction, `ai_extract`'s per-input pricing is more predictable for production.
